# 🔥 SignBridge - AUTSL 226 Sınıf Model Eğitimi

**Tek tuşla:** Runtime > Run all > Drive izni ver > Bekle (~1 saat)

- GPU: T4 seçili olmalı (Runtime > Change runtime type)
- Checkpoint: Her 5 epoch'ta Drive'a otomatik kaydeder
- Colab düşerse tekrar Run all — kaldığı yerden devam eder

In [ ]:
#@title 1️⃣ KURULUM + DRIVE + ESKİ CHECKPOINT TEMİZLİĞİ
import subprocess, sys, os, shutil

# Paketler
for pkg in ['torch', 'torchvision', 'numpy', 'tqdm']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('✅ Paketler hazır!')

# Drive bağla
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
print('✅ Drive bağlı!')

# ===== ESKİ BOZUK CHECKPOINT'LARI OTOMATİK SİL =====
eski_dir = '/content/drive/MyDrive/AUTSL_Proje/Models_v4_yeni_egitim'
if os.path.exists(eski_dir):
    for f in ['training_checkpoint.pt', 'best_model.pt']:
        fp = os.path.join(eski_dir, f)
        if os.path.exists(fp):
            os.remove(fp)
            print(f'🗑️ Silindi: {f}')
    print('✅ Eski checkpoint\'lar temizlendi!')
else:
    print('ℹ️ Eski checkpoint yok, temiz başlanıyor.')

In [ ]:
#@title 2️⃣ KONFİGÜRASYON + VERİ YÜKLEME
import numpy as np, json, csv, math, time, random, gc
from pathlib import Path
from collections import Counter
from tqdm import tqdm
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import matplotlib.pyplot as plt

class CONFIG:
    PROJE_DIR = '/content/drive/MyDrive/AUTSL_Proje'
    KOORDINAT_DIR = f'{PROJE_DIR}/Koordinatlar'
    MEVCUT_MODEL_DIR = f'{PROJE_DIR}/Models_v3_80plus'
    CLASS_LIST_CSV = f'{PROJE_DIR}/SignList_ClassId_TR_EN (1).csv'
    SAVE_DIR = f'{PROJE_DIR}/Models_v4_yeni_egitim'
    TRAIN_NPZ = f'{KOORDINAT_DIR}/all_train_packed.npz'
    VAL_NPZ = f'{KOORDINAT_DIR}/all_val_packed.npz'
    TEST_NPZ = f'{KOORDINAT_DIR}/all_test_packed.npz'
    INPUT_SIZE = 225; D_MODEL = 384; NHEAD = 12; NUM_LAYERS = 6
    NUM_CLASSES = 226; DROPOUT = 0.3; SEQ_LENGTH = 30
    EPOCHS = 60; BATCH_SIZE = 64; LR = 5e-5
    WEIGHT_DECAY = 0.01; WARMUP_EPOCHS = 3; LABEL_SMOOTHING = 0.1
    MIXUP_ALPHA = 0.2; GRADIENT_CLIP = 1.0; EMA_DECAY = 0.999
    USE_VELOCITY = False; USE_ACCELERATION = False; USE_RELATIVE_COORDS = False
    AUG_NOISE_STD = 0.01; AUG_MIRROR_PROB = 0.3
    AUG_SPEED_RANGE = (0.8, 1.2); AUG_SCALE_RANGE = (0.9, 1.1); AUG_DROPOUT_PROB = 0.1
    CHECKPOINT_EVERY = 5; RESUME_TRAINING = True; USE_PRETRAINED = True; PATIENCE = 15
cfg = CONFIG()

# Sınıf isimleri
class_names = {}
if os.path.exists(cfg.CLASS_LIST_CSV):
    with open(cfg.CLASS_LIST_CSV, 'r', encoding='utf-8') as f:
        reader = csv.reader(f); next(reader, None)
        for row in reader:
            if len(row) >= 2:
                try: class_names[int(row[0])] = row[1].strip()
                except: pass
    print(f'📋 {len(class_names)} sınıf ismi yüklendi')

# Veri yükle
data = {}
for split, fpath in [('train', cfg.TRAIN_NPZ), ('val', cfg.VAL_NPZ), ('test', cfg.TEST_NPZ)]:
    if not os.path.exists(fpath):
        print(f'❌ {split} bulunamadı: {fpath}'); continue
    d = np.load(fpath); keys = list(d.keys())
    X = d['x'].astype(np.float32) if 'x' in keys else d[keys[0]].astype(np.float32)
    y = d['y'].astype(np.int64) if 'y' in keys else d[keys[1]].astype(np.int64)
    data[split] = (X, y)
    print(f'✅ {split:5s}: {X.shape[0]:6d} örnek | shape={X.shape} | {len(set(y.tolist()))} sınıf')

X_train, y_train = data['train']
X_val, y_val = data['val']
X_test, y_test = data['test']

print(f'\n✅ Veri yüklendi!')

In [ ]:
#@title 3️⃣ AUGMENTATION + DATASET + MODEL

# ===== AUGMENTATION =====
def augment_sequence(seq, cfg):
    aug = seq.copy()
    T, F = aug.shape
    # Speed change
    if random.random() < 0.5:
        new_T = max(10, int(T * random.uniform(*cfg.AUG_SPEED_RANGE)))
        idx = np.clip(np.linspace(0, T-1, new_T).astype(int), 0, T-1)
        aug = aug[idx]
    # Temporal shift
    if random.random() < 0.3:
        aug = np.roll(aug, random.randint(-3, 3), axis=0)
    # Frame dropout
    if random.random() < 0.2:
        T2 = aug.shape[0]; result = aug.copy()
        for idx in random.sample(range(T2), max(1, int(T2 * 0.1))):
            p, n = max(0, idx-1), min(T2-1, idx+1)
            result[idx] = (aug[p] + aug[n]) / 2
        aug = result
    # Noise
    if random.random() < 0.5:
        aug = aug + np.random.randn(*aug.shape).astype(np.float32) * cfg.AUG_NOISE_STD
    # Mirror hands
    if random.random() < cfg.AUG_MIRROR_PROB:
        m = aug.copy()
        for i in range(0, 225, 3): m[:, i] = 1.0 - m[:, i]
        lh, rh = m[:, 99:162].copy(), m[:, 162:225].copy()
        m[:, 99:162], m[:, 162:225] = rh, lh
        aug = m
    # Scale
    if random.random() < 0.3:
        s = random.uniform(*cfg.AUG_SCALE_RANGE)
        for i in range(0, aug.shape[1], 3): aug[:, i] *= s; aug[:, i+1] *= s
    # Landmark dropout
    if random.random() < cfg.AUG_DROPOUT_PROB:
        n_lm = aug.shape[1] // 3
        for li in random.sample(range(n_lm), max(1, int(n_lm * 0.1))):
            aug[:, li*3:li*3+3] = 0.0
    return aug

def build_features(sequence, cfg):
    features = [sequence]
    if cfg.USE_VELOCITY:
        v = np.zeros_like(sequence); v[1:] = sequence[1:] - sequence[:-1]; features.append(v)
    if cfg.USE_ACCELERATION:
        a = np.zeros_like(sequence); a[2:] = sequence[2:] - 2*sequence[1:-1] + sequence[:-2]; features.append(a)
    return np.concatenate(features, axis=-1).astype(np.float32)

def pad_or_truncate(sequence, target_length):
    T = len(sequence)
    if T == target_length: return sequence
    elif T > target_length:
        s = (T - target_length) // 2; return sequence[s:s + target_length]
    else:
        return np.concatenate([sequence, np.tile(sequence[-1:], (target_length - T, 1))], axis=0)

def compute_feature_size(cfg):
    size = cfg.INPUT_SIZE
    if cfg.USE_VELOCITY: size += cfg.INPUT_SIZE
    if cfg.USE_ACCELERATION: size += cfg.INPUT_SIZE
    return size

class AUTSLDataset(Dataset):
    def __init__(self, X, y, cfg, is_train=True):
        self.X, self.y, self.cfg, self.is_train = X, y, cfg, is_train
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        seq = self.X[idx].copy()
        if self.is_train: seq = augment_sequence(seq, self.cfg)
        seq = pad_or_truncate(seq, self.cfg.SEQ_LENGTH)
        seq = build_features(seq, self.cfg)
        return torch.tensor(seq, dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.long)

# ===== MODEL =====
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class SignTransformerPro(nn.Module):
    def __init__(self, input_size, d_model=384, nhead=12, num_layers=6, num_classes=226, dropout=0.35):
        super().__init__()
        self.input_conv = nn.Sequential(
            nn.Linear(input_size, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout))
        self.conv_block = nn.Sequential(
            nn.Conv1d(d_model, d_model, 3, padding=1, groups=d_model),
            nn.Conv1d(d_model, d_model, 1),
            nn.BatchNorm1d(d_model), nn.GELU(), nn.Dropout(dropout))
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.pool_heads = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_model//4), nn.Tanh(), nn.Linear(d_model//4, 1))
            for _ in range(4)])
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model*5), nn.Dropout(dropout),
            nn.Linear(d_model*5, d_model*2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model*2, d_model), nn.GELU(), nn.Dropout(dropout/2),
            nn.Linear(d_model, num_classes))
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
    def forward(self, x):
        B = x.shape[0]
        x = self.input_conv(x)
        x = x + self.conv_block(x.transpose(1,2)).transpose(1,2)
        x = torch.cat([self.cls_token.expand(B,-1,-1), x], dim=1)
        x = self.transformer(self.pos_encoder(x))
        seq = x[:, 1:]
        pooled = [F.softmax(h(seq), dim=1) * seq for h in self.pool_heads]
        pooled = [p.sum(dim=1) for p in pooled]
        return self.classifier(torch.cat(pooled + [seq.mean(dim=1)], dim=1))

# ===== EMA =====
class EMAModel:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {}
        for name, param in model.named_parameters():
            if param.requires_grad: self.shadow[name] = param.data.clone()
    def update(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = self.decay * self.shadow[name] + (1 - self.decay) * param.data
    def apply(self, model):
        backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                backup[name] = param.data.clone(); param.data = self.shadow[name]
        return backup
    def restore(self, model, backup):
        for name, param in model.named_parameters():
            if param.requires_grad and name in backup: param.data = backup[name]
    def state_dict(self): return dict(self.shadow)
    def load_state_dict(self, sd): self.shadow = {k: v.clone() for k, v in sd.items()}

class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_epochs, total_epochs, min_lr=1e-6):
        self.optimizer, self.warmup = optimizer, warmup_epochs
        self.total, self.min_lr = total_epochs, min_lr
        self.base_lrs = [pg['lr'] for pg in optimizer.param_groups]
    def step(self, epoch):
        if epoch < self.warmup:
            p = epoch / max(1, self.warmup)
            for pg, blr in zip(self.optimizer.param_groups, self.base_lrs): pg['lr'] = blr * p
        else:
            p = (epoch - self.warmup) / max(1, self.total - self.warmup)
            for pg, blr in zip(self.optimizer.param_groups, self.base_lrs):
                pg['lr'] = self.min_lr + (blr - self.min_lr) * 0.5 * (1 + math.cos(math.pi * p))

def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

print('✅ Model + Dataset + Augmentation hazır!')

In [ ]:
#@title 4️⃣ EĞİTİM (Ana Döngü)

cfg = CONFIG()
os.makedirs(cfg.SAVE_DIR, exist_ok=True)
feature_size = compute_feature_size(cfg)
print(f'Feature size: {feature_size}')

# Normalization
print('Normalization hesaplanıyor...')
all_features = []
for i in range(0, len(X_train), 500):
    batch = X_train[i:i+500]
    for seq in batch:
        seq_p = pad_or_truncate(seq, cfg.SEQ_LENGTH)
        feat = build_features(seq_p, cfg)
        all_features.append(feat)
all_features = np.concatenate(all_features, axis=0)
norm_mean = all_features.mean(axis=0)
norm_std = np.maximum(all_features.std(axis=0), 1e-6)
print(f'  norm_mean shape: {norm_mean.shape}')
print(f'  norm_std  shape: {norm_std.shape}')
del all_features; gc.collect()

# Datasets & Loaders
train_dataset = AUTSLDataset(X_train, y_train, cfg, is_train=True)
val_dataset = AUTSLDataset(X_val, y_val, cfg, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
norm_mean_t = torch.tensor(norm_mean, dtype=torch.float32).to(device)
norm_std_t = torch.tensor(norm_std, dtype=torch.float32).to(device)
print(f'Device: {device}')

# Model
model = SignTransformerPro(
    input_size=feature_size,
    d_model=cfg.D_MODEL, nhead=cfg.NHEAD,
    num_layers=cfg.NUM_LAYERS, num_classes=cfg.NUM_CLASSES,
    dropout=cfg.DROPOUT
).to(device)

# Load pretrained
pretrained_path = os.path.join(cfg.PROJE_DIR, 'Models_v3_80plus', 'best_model.pt')
if os.path.exists(pretrained_path):
    state = torch.load(pretrained_path, map_location=device, weights_only=True)
    if isinstance(state, dict) and 'model_state_dict' in state:
        model.load_state_dict(state['model_state_dict'], strict=False)
        print(f'✅ Pretrained yüklendi (val_acc: {state.get("val_acc", "?")})' )
    else:
        model.load_state_dict(state, strict=False)
        print('✅ Pretrained yüklendi (raw state_dict)')
else:
    print(f'⚠️ Pretrained bulunamadı: {pretrained_path}')

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,} | Trainable: {trainable_params:,}')

# Optimizer, scheduler, EMA
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
scheduler = WarmupCosineScheduler(optimizer, cfg.WARMUP_EPOCHS, cfg.EPOCHS, min_lr=1e-6)
criterion = nn.CrossEntropyLoss(label_smoothing=cfg.LABEL_SMOOTHING)
ema = EMAModel(model, decay=cfg.EMA_DECAY)
scaler = torch.amp.GradScaler('cuda')

# Check existing checkpoint
checkpoint_loaded = False
start_epoch = 0
best_val_acc = 0.0
history = {'train_loss': [], 'val_acc': [], 'lr': []}

checkpoint_path = os.path.join(cfg.SAVE_DIR, 'checkpoint.pt')
if os.path.exists(checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    ckpt_acc = ckpt.get('val_acc', 0)
    if ckpt_acc < 50.0:
        print(f'⚠️ Checkpoint kalitesiz (val_acc={ckpt_acc:.2f}%), siliniyor...')
        os.remove(checkpoint_path)
    else:
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        start_epoch = ckpt['epoch'] + 1
        best_val_acc = ckpt_acc
        if 'ema_state_dict' in ckpt:
            ema.load_state_dict(ckpt['ema_state_dict'])
        if 'history' in ckpt:
            history = ckpt['history']
        checkpoint_loaded = True
        print(f'✅ Checkpoint yüklendi: epoch {start_epoch}, val_acc={best_val_acc:.2f}%')

if not checkpoint_loaded:
    print(f'🆕 Sıfırdan eğitim (pretrained üzerinden fine-tuning)')

# ===== TRAINING LOOP =====
patience_counter = 0

for epoch in range(start_epoch, cfg.EPOCHS):
    scheduler.step(epoch)
    current_lr = optimizer.param_groups[0]['lr']

    # --- Train ---
    model.train()
    total_loss, correct, total = 0, 0, 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{cfg.EPOCHS}')

    for batch_x, batch_y in pbar:
        batch_x = (batch_x.to(device) - norm_mean_t) / norm_std_t
        batch_y = batch_y.to(device)

        if cfg.MIXUP_ALPHA > 0 and random.random() < 0.5:
            lam = np.random.beta(cfg.MIXUP_ALPHA, cfg.MIXUP_ALPHA)
            idx = torch.randperm(batch_x.size(0), device=device)
            mixed_x = lam * batch_x + (1 - lam) * batch_x[idx]
            with torch.amp.autocast('cuda'):
                out = model(mixed_x)
                loss = lam * criterion(out, batch_y) + (1 - lam) * criterion(out, batch_y[idx])
        else:
            with torch.amp.autocast('cuda'):
                out = model(batch_x)
                loss = criterion(out, batch_y)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        ema.update(model)

        total_loss += loss.item() * batch_y.size(0)
        correct += (out.argmax(1) == batch_y).sum().item()
        total += batch_y.size(0)
        pbar.set_postfix(loss=f'{loss.item():.4f}', acc=f'{100*correct/total:.1f}%', lr=f'{current_lr:.2e}')

    train_loss = total_loss / total
    train_acc = 100 * correct / total

    # --- Validation with EMA ---
    ema_backup = ema.apply(model)
    model.eval()
    val_correct, val_total = 0, 0

    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x = (batch_x.to(device) - norm_mean_t) / norm_std_t
            batch_y = batch_y.to(device)
            with torch.amp.autocast('cuda'):
                out = model(batch_x)
            val_correct += (out.argmax(1) == batch_y).sum().item()
            val_total += batch_y.size(0)

    val_acc = 100 * val_correct / val_total
    ema.restore(model, ema_backup)

    history['train_loss'].append(train_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)

    print(f'  \U0001f4ca Epoch {epoch+1}: train_loss={train_loss:.4f}, train_acc={train_acc:.1f}%, val_acc={val_acc:.2f}%, lr={current_lr:.2e}')

    # Save best
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0

        ema_backup2 = ema.apply(model)
        torch.save({
            'model_state_dict': model.state_dict(),
            'val_acc': val_acc, 'epoch': epoch,
            'config': {k: v for k, v in vars(cfg).items() if not k.startswith('_')},
        }, os.path.join(cfg.SAVE_DIR, 'best_model.pt'))
        ema.restore(model, ema_backup2)
        print(f'  \U0001f3c6 Yeni best: {val_acc:.2f}%')
    else:
        patience_counter += 1

    # Save checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'ema_state_dict': ema.state_dict(),
            'epoch': epoch, 'val_acc': val_acc,
            'best_val_acc': best_val_acc,
            'history': history,
        }, checkpoint_path)
        print(f'  \U0001f4be Checkpoint kaydedildi (epoch {epoch+1})')

    # Early stopping
    if patience_counter >= cfg.PATIENCE:
        print(f'  \u23f9\ufe0f Early stopping ({cfg.PATIENCE} epoch iyile\u015fme yok)')
        break

print(f'\n\U0001f389 E\u011fitim tamamland\u0131! Best val_acc: {best_val_acc:.2f}%')

In [ ]:
#@title 5️⃣ TEST + KAYIT + ANALİZ

# ===== TEST SET EVALUATION =====
print('='*60)
print('TEST SET DE\u011eERLEND\u0130RME')
print('='*60)

# Load best model
best_path = os.path.join(cfg.SAVE_DIR, 'best_model.pt')
if os.path.exists(best_path):
    best_state = torch.load(best_path, map_location=device, weights_only=True)
    if 'model_state_dict' in best_state:
        model.load_state_dict(best_state['model_state_dict'])
    else:
        model.load_state_dict(best_state)
    print(f'✅ Best model yüklendi (val_acc: {best_state.get("val_acc", "?")})' )

test_dataset = AUTSLDataset(X_test, y_test, cfg, is_train=False)
test_loader = DataLoader(test_dataset, batch_size=cfg.BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)

model.eval()
test_correct, test_total = 0, 0
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for batch_x, batch_y in tqdm(test_loader, desc='Test'):
        batch_x = (batch_x.to(device) - norm_mean_t) / norm_std_t
        batch_y = batch_y.to(device)
        with torch.amp.autocast('cuda'):
            out = model(batch_x)
        probs = F.softmax(out, dim=1)
        preds = out.argmax(1)
        test_correct += (preds == batch_y).sum().item()
        test_total += batch_y.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch_y.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

test_acc = 100 * test_correct / test_total
print(f'\n\U0001f3af Test Accuracy: {test_acc:.2f}% ({test_correct}/{test_total})')

# ===== SAVE NORMALIZATION + LABEL MAP =====
np.save(os.path.join(cfg.SAVE_DIR, 'norm_mean.npy'), norm_mean)
np.save(os.path.join(cfg.SAVE_DIR, 'norm_std.npy'), norm_std)
print(f'✅ norm_mean.npy, norm_std.npy kaydedildi')

label_map = {str(i): class_names[i] for i in range(len(class_names))}
with open(os.path.join(cfg.SAVE_DIR, 'label_map.json'), 'w', encoding='utf-8') as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)
print(f'✅ label_map.json kaydedildi ({len(label_map)} sınıf)')

# Save EMA model too
ema_backup3 = ema.apply(model)
torch.save({
    'model_state_dict': model.state_dict(),
    'val_acc': best_val_acc,
    'test_acc': test_acc,
}, os.path.join(cfg.SAVE_DIR, 'ema_model.pt'))
ema.restore(model, ema_backup3)
print(f'✅ EMA model kaydedildi')

# ===== TRAINING CURVES =====
if len(history['val_acc']) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].plot(history['train_loss'], 'b-', linewidth=2)
    axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].grid(True)
    axes[1].plot(history['val_acc'], 'r-', linewidth=2)
    axes[1].axhline(y=best_val_acc, color='g', linestyle='--', label=f'Best: {best_val_acc:.2f}%')
    axes[1].set_title('Validation Accuracy (%)'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True)
    axes[2].plot(history['lr'], 'g-', linewidth=2)
    axes[2].set_title('Learning Rate'); axes[2].set_xlabel('Epoch'); axes[2].grid(True)
    plt.tight_layout(); plt.savefig(os.path.join(cfg.SAVE_DIR, 'training_curves.png'), dpi=150); plt.show()

# ===== PER-CLASS ANALYSIS =====
all_preds_np = np.array(all_preds)
all_labels_np = np.array(all_labels)
all_probs_np = np.array(all_probs)

print('\n' + '='*60)
print('SINIF BAZLI ANAL\u0130Z')
print('='*60)

per_class_correct = {}
per_class_total = {}
for p, l in zip(all_preds_np, all_labels_np):
    per_class_total[l] = per_class_total.get(l, 0) + 1
    if p == l: per_class_correct[l] = per_class_correct.get(l, 0) + 1

# Worst classes
worst_classes = []
for cls_id in per_class_total:
    acc = 100 * per_class_correct.get(cls_id, 0) / per_class_total[cls_id]
    name = class_names[cls_id] if cls_id < len(class_names) else f'class_{cls_id}'
    worst_classes.append((cls_id, name, acc, per_class_total[cls_id]))

worst_classes.sort(key=lambda x: x[2])
print('\n\U0001f534 En d\u00fc\u015f\u00fck 20 s\u0131n\u0131f:')
for cls_id, name, acc, n in worst_classes[:20]:
    print(f'  {cls_id:>4} {name:<30} {acc:>6.1f}% (n={n})')

# Best classes
print('\n\U0001f7e2 En y\u00fcksek 10 s\u0131n\u0131f:')
for cls_id, name, acc, n in sorted(worst_classes, key=lambda x: -x[2])[:10]:
    print(f'  {cls_id:>4} {name:<30} {acc:>6.1f}% (n={n})')

# Confusion pairs
print('\n\U0001f504 En \u00e7ok kar\u0131\u015fan s\u0131n\u0131f \u00e7iftleri:')
confusion_pairs = {}
for p, l in zip(all_preds_np, all_labels_np):
    if p != l:
        pair = (int(l), int(p))
        confusion_pairs[pair] = confusion_pairs.get(pair, 0) + 1

sorted_pairs = sorted(confusion_pairs.items(), key=lambda x: -x[1])[:15]
for (true_cls, pred_cls), count in sorted_pairs:
    true_name = class_names[true_cls] if true_cls < len(class_names) else f'class_{true_cls}'
    pred_name = class_names[pred_cls] if pred_cls < len(class_names) else f'class_{pred_cls}'
    print(f'  {true_name} -> {pred_name}: {count} kez')

# Top-5 accuracy
all_probs_t = torch.tensor(all_probs_np)
all_labels_t = torch.tensor(all_labels_np)
top5_preds = all_probs_t.topk(5, dim=1).indices
top5_correct = sum(1 for i in range(len(all_labels_t)) if all_labels_t[i] in top5_preds[i])
top5_acc = 100 * top5_correct / len(all_labels_t)

# Confidence stats
correct_mask = all_preds_np == all_labels_np
correct_probs = all_probs_np[np.arange(len(all_preds_np)), all_preds_np][correct_mask]
wrong_probs = all_probs_np[np.arange(len(all_preds_np)), all_preds_np][~correct_mask]

print(f'\n\U0001f4ca OZET:')
print(f'  Test Accuracy (Top-1): {test_acc:.2f}%')
print(f'  Test Accuracy (Top-5): {top5_acc:.2f}%')
print(f'  Best Val Accuracy:     {best_val_acc:.2f}%')
print(f'  Dogru tahmin avg conf: {correct_probs.mean():.4f}')
if len(wrong_probs) > 0:
    print(f'  Yanlis tahmin avg conf: {wrong_probs.mean():.4f}')
print(f'  Toplam epoch:          {len(history["val_acc"])}')

# List saved files
print(f'\n\U0001f4c1 Kaydedilen dosyalar ({cfg.SAVE_DIR}):')
for f in sorted(os.listdir(cfg.SAVE_DIR)):
    fpath = os.path.join(cfg.SAVE_DIR, f)
    size = os.path.getsize(fpath) / (1024*1024)
    print(f'  {f} ({size:.1f} MB)')

print('\n\u2705 Tum islemler tamamlandi!')
print(f'\U0001f3af Modeli kullanmak icin {cfg.SAVE_DIR}/best_model.pt dosyasini indirin.')